# Portfolio Optimization — Stateless Multi-Asset Notebook Adapter

This sanitized notebook delegates all calculations to the canonical `portfolio_optimization` package (`src/portfolio_optimization/`).

- Stateless execution: all configuration and results live strictly in memory per run (no accounts, no persistence).
- Optional server-side risk-free fallback: `FRED_API_KEY` via environment variable or local `.env` (never commit secrets).
- No outputs, widget states, or API keys are stored in this file.


In [ ]:
import portfolio_optimization as po

po.configure_runtime(print_banner=True)


# Portfolio Configuration (Variable Income + Fixed Income)

In [ ]:
config = po.PortfolioAnalysisConfig(
    variable_income=(
        po.InstrumentConfig(
            instrument_id="AAPL",
            ticker="AAPL",
            section="variable_income",
            asset_subtype="equity",
            dcf_view=po.DCFViewConfig(
                valuation_basis="terminal_target",
                total_upside=0.50,
                horizon_years=5.0,
                confidence=0.60,
            ),
        ),
        po.InstrumentConfig(instrument_id="MSFT", ticker="MSFT", section="variable_income", asset_subtype="equity"),
        po.InstrumentConfig(instrument_id="GLD", ticker="GLD", section="variable_income", asset_subtype="commodity_gold"),
        po.InstrumentConfig(instrument_id="BTC-USD", ticker="BTC-USD", section="variable_income", asset_subtype="crypto"),
    ),
    fixed_income=(
        po.InstrumentConfig(instrument_id="SHY", ticker="SHY", section="fixed_income", asset_subtype="fixed_income_etf"),
    ),
    constraints=po.ConstraintsConfig(
        class_bounds={"crypto": (0.0, 0.10)},
    ),
    optimization=po.OptimizationConfig(selected_strategy="erc"),
)


# Run Analysis

In [ ]:
result = po.ejecutar_analisis_completo(config, render_ui=True)


# Analytics & Indicative Rebalancing

In [ ]:
po.generar_frontera_eficiente(result, n_samples=2000)
po.analizar_rendimiento_historico(result)
po.generar_heatmap_correlaciones(result)
po.generar_underwater_plot(result)
po.analizar_contribucion_marginal_riesgo(result)
po.generar_grafico_riesgo(result)
po.mostrar_rebalanceo(result)
allocations = po.armar_cartera(result, capital_disponible=10000.0)
